In [16]:
from dotenv import load_dotenv
from datetime import datetime, timedelta

import os
import json
import openai
import requests

# Load environment variables from .env file
load_dotenv('~/Documents/Jupyter/MDST/WN25/Eventure/EventureApp/.env')

openai.api_key = os.getenv("OPENAI_API_KEY")
ticketmaster_key = os.getenv("TICKETMASTER_API_KEY")

# Verify
print("OpenAI Key Found:", bool(os.getenv("OPENAI_API_KEY")))

MODEL_NAME = "gpt-4o-mini"



OpenAI Key Found: True


Create Functions:

In [ ]:
# def makeTicketmasterEventRequest(eventparams: str):
#     # Format for looking at events: https://app.ticketmaster.com/{package}/{version}/events.json?{eventparams}apikey={API key}
#     # eventparams needs to be separated by & symbols for each query. Check ticketmaster discovery page to see all query options

#     url = f'https://app.ticketmaster.com/discovery/v2/events.json?{eventparams}&apikey={ticketmaster_key}'

#     r = requests.get(url)

#     return r 

def makeTicketmasterEventRequest(params: dict):
    params['apikey'] = ticketmaster_key
    url = f'https://app.ticketmaster.com/discovery/v2/events.json'

    r = requests.get(url,params=params)

    return r 

Describe functions to OpenAI:

In [21]:
function_descriptions = [
    {
        "type": "function",
        "function": {
            "name": "makeTicketmasterEventRequest",
            "description": "Fetches events happening in a specific location from Ticketmaster.",
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {
                        "type": "string",
                        "description": "Search term for events, e.g., 'music', 'sports'."
                    },
                    "city": {
                        "type": "string",
                        "description": "The city to fetch events for, e.g., 'Detroit'."
                    },
                    "stateCode": {
                        "type": "string",
                        "description": "The state code, e.g., 'MI' for Michigan."
                    },
                    "countryCode": {
                        "type": "string",
                        "description": "The country code, e.g., 'US'."
                    },
                    "startDateTime": {
                        "type": "string",
                        "format": "date-time",
                        "description": "Start date and time for events in ISO 8601 format (YYYY-MM-DDTHH:MM:SSZ)."
                    },
                    "endDateTime": {
                        "type": "string",
                        "format": "date-time",
                        "description": "End date and time for events in ISO 8601 format."
                    }
                },
                "required": ["city", "stateCode", "countryCode"]
            }
        }
    }
]


In [22]:
def test_call_model(user_message: str, function_call="auto"):
    """
    1) We send user_message + function_descriptions to the model
    2) We see if it returns a function call
    3) If so, we parse arguments, call the function ourselves
    4) Return final output
    """
    client = openai.OpenAI()

    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": user_message}],
        tools=function_descriptions,  # 'functions' is now called 'tools'
        tool_choice=function_call,    # 'function_call' is now 'tool_choice'
    )
    response = completion.choices[0].message
    return response

In [ ]:
# Let's do a quick test with something that calls 'get_weather_info'
user_prompt_1 = "What events are going on in Michigan for this month"
resp = test_call_model(user_prompt_1, function_call="auto")
print("Model response:\n", resp)


Model response:
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_w34zZU8EeUNzTcmK7WCBt5Mn', function=Function(arguments='{"city":"Detroit","stateCode":"MI","countryCode":"US"}', name='makeTicketmasterEventRequest'), type='function')])
